# 03 — Price-change events

This notebook presents **observed price-change candidates** over the validated
pricing population: how listed prices moved from each scheduled capture to the
immediately following one, whether movements were isolated or synchronized,
how airport and downtown locations moved together, whether new prices
persisted, and the derived final Vancouver decrease.

It is a thin presentation layer. Every event, synchronization rule,
persistence match, case selection, reconciliation and privacy rule comes from
`ql2_sixt_canada_analysis` (`run_price_change_presentation` →
`run_pricing_pipeline` → event engine → higher-order analysis). It does not
depend on notebooks 01 or 02 or on earlier kernel state.

The results are descriptive. They do not prove genuine repricing, and they do
not prove extraction anomalies.

## Data-plan scope

Price-change events, presented here:

- ordering by the canonical scheduled capture timestamp;
- matching each offer to the immediately preceding scheduled capture;
- absolute (exact-cent) and percentage changes;
- separating price changes from appearances and disappearances;
- synchronized movements and airport/downtown co-movement;
- the final Vancouver decrease;
- persistence when a following eligible capture exists, with right-censored
  final captures kept apart;
- the event table and the event heatmap.

Out of scope: the *Visible assortment stability* story (appearance and
disappearance counts appear only where they are needed to interpret price
changes) and *Monitoring and actionability* (no thresholds, alerts, baselines
or production rules).

## Confidentiality

All QL2 data is proprietary. This notebook shows only sanitized aggregate
tables and the aggregate heatmap. It never shows event rows, product
identities, individual prices, raw identifiers, file names or paths. The
local detailed event table is never displayed. Clear all outputs before
committing (Edit ▸ Clear Outputs of All Cells).

In [ ]:
from IPython.display import Image, display

from ql2_sixt_canada_analysis import heatmap_png, presentation_settings, run_price_change_presentation
from ql2_sixt_canada_analysis.paths import resolve_raw_data_dir

# Raw-data directory: the centralized default unless the documented environment
# override is set. Set RAW_DIR_OVERRIDE only for a deliberate local experiment.
RAW_DIR_OVERRIDE = None
# Optional explicit, Git-ignored output directory for the local artifacts. None writes
# nothing; the documented environment variables can set it instead.
OUTPUT_DIR = None
# Opt in to the confidential local detailed event table (needs an output directory).
WRITE_DETAIL = None

raw_dir = resolve_raw_data_dir(RAW_DIR_OVERRIDE)
output_dir, write_detail = presentation_settings(OUTPUT_DIR, WRITE_DETAIL)
print("Raw-data directory selected.")
print("Local artifacts requested:", output_dir is not None)
print("Local detailed export requested:", write_detail)

## Pipeline and readiness

One call runs the pricing pipeline exactly once and passes that same result
through the event engine, the higher-order analysis and every reconciliation
gate. If anything is blocked, only the blocker categories are shown and
nothing else is calculated.

In [ ]:
presentation = run_price_change_presentation(raw_dir, output_dir=output_dir, write_detail=write_detail)
ready = presentation.completed
print("Price-change presentation status:", presentation.report.status.value)
if not ready:
    print("Blocked - no tables, figure or local output are produced.")
    print("Blockers:", ", ".join(b.value for b in presentation.report.blockers))
    if presentation.report.upstream_blockers:
        print("Upstream blockers:", ", ".join(presentation.report.upstream_blockers))

**Interpretation.** A `completed` status means the event engine, the higher-order analysis and every presentation check ran on one pipeline result. A `blocked` status means no price-change finding is valid; the blocker categories name the failing gate.

## Event population and reconciliation

Every presentation table is reconciled with the validated event report
before it is shown. The checks include that increases plus decreases equal
the price-change population, that appearances plus disappearances equal the
assortment-event population, that persistence partitions every price change
exactly once, and that the heatmap source equals the interval table.

In [ ]:
if ready:
    tables = presentation.tables
    print("All presentation reconciliation checks passed:", presentation.report.reconciled)
    display(tables.reconciliation_summary)
else:
    print("Skipped: the presentation is blocked.")

**Interpretation.** Every check must read `reconciled`: increases plus decreases equal the price changes, and persistence partitions every change exactly once. A failed check blocks the presentation instead of showing inconsistent tables.

## Synchronized movements

One row per approved canonical location and eligible one-hour interval,
including quiet intervals. Three synchronization notions are kept separate,
and each needs at least two changed offers:

- **direction**: every changed offer moved the same way (unchanged offers stay
  in the comparable denominator);
- **exact cent**: every changed offer has the same signed cent change;
- **exact percentage**: every changed offer has a nonzero previous price and
  the same exact rational percentage.

An interval is a *material synchronized movement* only under the documented
descriptive rule (direction-synchronized). This is not an alert threshold.
Isolated and mixed-direction movements are shown as excluded, so selected
plus excluded price changes add up to all price changes. Magnitude summaries
are suppressed by the package when their own contributors are too few: cent summaries count
changed offers and percentage summaries count changed offers with a nonzero previous price.

In [ ]:
if ready:
    display(tables.event_interval_summary)
    display(tables.material_synchronized_movements)
    display(tables.material_selection_reconciliation)
else:
    print("Skipped: the presentation is blocked.")

**Interpretation.** Direction-synchronized intervals are hours in which several products at one location moved the same way: the pattern most worth reviewing with the collection owner and revenue management. They are candidates, not proof of intentional repricing; isolated and mixed movements are listed as excluded so the totals reconcile.

## Airport/downtown comparisons

Pairs and roles come only from the approved location authority, never from
location names. Within one city and one exact interval, products are matched
on the full product and unit identity. Same direction, exact cent and exact
percentage are reported separately. A product seen at only one location is
counted, not compared.

In [ ]:
if ready:
    display(tables.airport_downtown_summary)
else:
    print("Skipped: the presentation is blocked.")

**Interpretation.** Same-direction counts show whether airport and downtown prices of the same product moved together within one hour. Co-movement is descriptive; products seen at only one location are counted, not compared.

## Persistence

Each increase or decrease is judged only at the immediately following
eligible interval: held, continued, reverted, disappeared or ambiguous.
*Not testable* covers right-censored final captures and hard breaks, kept
apart by reason. Price-path shares use held + continued + reverted as the
denominator. Disappeared and ambiguous shares use events with a following
interval. Not-testable events are never in a denominator.

In [ ]:
if ready:
    display(tables.persistence_summary)
    summary = presentation.analysis.report.persistence
    print("Changed events with a following eligible interval:", summary.with_following_interval)
    print("Not testable (right-censored or hard break):", summary.not_testable)
    for reason, count in summary.not_testable_reasons:
        print(f"  {reason}: {count}")
else:
    print("Skipped: the presentation is blocked.")

**Interpretation.** Held and continued changes kept the new price at the next eligible capture; reverted changes moved back. Right-censored and break-affected changes are not testable and are in no share, so persistence describes the next hour only.

## Final Vancouver decrease

The case is derived, never hard-coded: it is the latest eligible interval of
the governed alias city with at least one decrease. Vancouver Downtown and
Thurlow form one canonical location, and source aliases are provenance only.

The table is one fixed-schema aggregate record: counts, flags, enum statuses,
canonical scheduled periods and magnitude summaries. Each magnitude summary is governed by its
own contributors (decreases for cents, percent-valid decreases for percentages) and shows as
unavailable, never zero, when the package suppresses it. Alias
provenance appears only as fixed category counts, never as source labels, and the
descriptive sentence is printed separately instead of being stored in the table.

In [ ]:
if ready:
    display(tables.final_vancouver_decrease)
    case = presentation.analysis.report.final_decrease
    print(case.describe())
    if case.status.value == "derived" and not case.persistence_testable:
        print("Persistence is not testable: the decrease occurs at the final capture (right censoring).")
else:
    print("Skipped: the presentation is blocked.")

**Interpretation.** This derived case is the latest Vancouver interval with a decrease. Its indicators help judge whether it looks more like a broad repricing candidate or an extraction issue; at the final capture its persistence cannot be tested and needs the next collection.

## Event heatmap

Increases and decreases are shown in separate panels on one common count
scale, so opposite movements never cancel. Every eligible interval is shown,
including quiet ones. Hatched grey cells have no eligible interval
(governed exclusion, missing capture, non-one-hour gap, source-stream change,
first capture or outside the schedule) and are not zero activity. The image
is rendered in memory from the reconciled source.

In [ ]:
if ready:
    display(Image(data=heatmap_png(presentation.analysis)))
else:
    print("Skipped: the presentation is blocked.")

**Interpretation.** Many increase or decrease cells across locations in the same hour indicate synchronized activity worth review; hatched cells have no eligible interval and are not zero activity.

## Limitations and interpretation

- The collection spans roughly 90 hours, and findings are descriptive.
- Synchronized observations are candidates, not proof of intentional
  repricing.
- Right-censored events provide no persistence evidence.
- Source and operational corroboration would be required to distinguish
  genuine repricing from extraction anomalies.
- Nothing here establishes production monitoring thresholds or long-term
  behaviour, and hourly captures are repeated measurements of the same
  products.

## Local detailed export

The confidential detailed event table (Parquet, one row per validated
candidate) is written only when an explicit output directory is set and the
detailed export is enabled: `OUTPUT_DIR` / `WRITE_DETAIL` above, or the
documented environment variables. Use a Git-ignored location such as
`reports/price_change_events/`. It is never displayed here, and the export
refuses raw-data directories.

In [ ]:
if ready and presentation.report.artifacts_written:
    print("Local artifacts written:", ", ".join(presentation.report.artifacts_written))
else:
    print("No files written.")